# Transfer Learning Introduction

Transfer learning is a machine learning technique where knowledge gained from training one model on a specific task is applied to a different but related task. This approach allows us to leverage pre-trained models to solve new problems, especially when we have limited data or computational resources.

This notebook explores the concepts and implementation of transfer learning in deep learning.

## 1. Import Required Libraries

In [ ]:
# TensorFlow and Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from torchvision import models

# Data manipulation and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Utilities
import os
import random
import time
from PIL import Image

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
torch.manual_seed(42)
random.seed(42)

print("TensorFlow version:", tf.__version__)
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

## 2. Understanding Transfer Learning

Transfer learning is a machine learning methodology where a pre-trained model developed for one task is reused as the starting point for a model on a second task. It's particularly valuable when you have a small dataset for your target task.

### Why Use Transfer Learning?

1. **Efficient Learning**: Leverages knowledge from pre-trained models, reducing training time and computational resources.
2. **Improved Performance**: Often results in better performance than training models from scratch, especially with limited data.
3. **Reduced Overfitting**: Pre-trained models have learned robust, generalizable features that help prevent overfitting on small datasets.

### When to Use Transfer Learning?

1. **Limited Training Data**: When you don't have enough labeled data to train a complex model from scratch.
2. **Similar Tasks**: When the source task (pre-trained model) shares features or patterns with your target task.
3. **Resource Constraints**: When you have limited computational resources or time for training.

### Transfer Learning Approaches

1. **Feature Extraction**: Use a pre-trained model as a feature extractor by keeping the convolutional base intact and training only new classifier layers.
2. **Fine-tuning**: Modify and retrain parts of the pre-trained model to better suit your specific task.

### Transfer Learning Visualization

Here's a conceptual diagram of how transfer learning works:

In [ ]:
# Create a simple visualization of transfer learning concept
plt.figure(figsize=(12, 6))

# Create a diagram showing transfer learning flow
plt.subplot(1, 1, 1)
plt.axis('off')
plt.text(0.1, 0.8, "Pre-trained Model\n(e.g., ImageNet)", fontsize=14, bbox=dict(facecolor='lightblue', alpha=0.5))
plt.text(0.7, 0.8, "Target Model", fontsize=14, bbox=dict(facecolor='lightgreen', alpha=0.5))

plt.arrow(0.25, 0.8, 0.4, 0, head_width=0.04, head_length=0.02, fc='black', ec='black')

plt.text(0.05, 0.6, "Trained on large dataset\n(e.g., 1.2M images)", fontsize=12)
plt.text(0.65, 0.6, "Trained on small dataset\n(e.g., hundreds of images)", fontsize=12)

plt.text(0.425, 0.85, "Transfer", fontsize=12)

# Add descriptions for approaches
plt.text(0.05, 0.4, "Approach 1: Feature Extraction", fontsize=14, fontweight='bold')
plt.text(0.05, 0.35, "- Freeze pre-trained layers\n- Replace & train only classifier", fontsize=12)

plt.text(0.05, 0.2, "Approach 2: Fine-Tuning", fontsize=14, fontweight='bold')
plt.text(0.05, 0.15, "- Freeze early layers\n- Retrain later layers\n- Train new classifier", fontsize=12)

plt.text(0.5, 0.3, "Benefits:\n- Better performance\n- Less training data\n- Faster convergence", 
         fontsize=13, bbox=dict(facecolor='lightyellow', alpha=0.5))

plt.tight_layout()
plt.show()

## 3. Pre-trained Model Architectures

Let's explore some popular pre-trained model architectures that are commonly used for transfer learning:

### VGG16/19
- Developed by Visual Geometry Group at Oxford
- Simple architecture with multiple stacked convolutional layers
- 16 or 19 layers deep
- Good feature extractors but relatively large models

### ResNet (Residual Networks)
- Introduced the concept of "skip connections" to address vanishing gradient problems
- Available in various depths (ResNet18, ResNet34, ResNet50, ResNet101, ResNet152)
- Excellent performance with reasonable model size

### MobileNet
- Designed for mobile and edge devices
- Uses depth-wise separable convolutions to reduce model size and computation
- Much smaller and faster than VGG or ResNet, with moderate accuracy trade-off

### Inception (GoogleNet)
- Uses inception modules with multiple filter sizes
- Efficient use of computing resources
- Good balance between accuracy and complexity

### EfficientNet
- Uses compound scaling to balance network depth, width, and resolution
- State-of-the-art performance with various model sizes (B0 to B7)
- Very efficient parameter usage

### The ImageNet Dataset
Most of these models were pre-trained on the ImageNet dataset, which contains:
- 1.2+ million images
- 1,000 different categories
- Diverse set of objects, animals, scenes, etc.

In [ ]:
# Let's visualize the architectures of some popular pre-trained models
def plot_model_diagram(name, layers_info):
    """
    Create a simple visualization of model architecture
    layers_info is a list of tuples (name, color, layers)
    """
    plt.figure(figsize=(10, 4))
    plt.axis('off')
    plt.title(f"{name} Architecture", fontsize=16)
    
    y_pos = 0.8
    for i, (section_name, color, num_layers) in enumerate(layers_info):
        x_start = 0.1 + i * 0.2
        plt.rectangle = plt.Rectangle((x_start, y_pos-0.2), 0.15, 0.2, fc=color, alpha=0.7)
        plt.gca().add_patch(plt.rectangle)
        plt.text(x_start + 0.075, y_pos-0.1, f"{num_layers}", ha='center')
        plt.text(x_start + 0.075, y_pos-0.25, section_name, ha='center', fontsize=10)
    
    plt.arrow(0.1, y_pos-0.1, 0.8, 0, head_width=0.02, head_length=0.01, fc='black', ec='black')
    plt.text(0.5, y_pos-0.35, "Forward Direction", ha='center')
    plt.text(0.1, y_pos+0.05, "Input", ha='center')
    plt.text(0.9, y_pos+0.05, "Output", ha='center')

# Visualize architectures
models_to_visualize = [
    ("VGG16", [
        ("Conv Blocks", "lightblue", "13 Conv\nLayers"),
        ("FC", "lightgreen", "3 FC\nLayers")
    ]),
    ("ResNet50", [
        ("Conv", "lightblue", "1 Conv"),
        ("ResBlocks", "lightblue", "16 Res\nBlocks"),
        ("FC", "lightgreen", "1 FC")
    ]),
    ("MobileNetV2", [
        ("Conv", "lightblue", "1 Conv"),
        ("Bottleneck", "lightcoral", "17 Bottleneck\nBlocks"),
        ("Conv+Pool", "lightblue", "Conv+\nPool"),
        ("FC", "lightgreen", "1 FC")
    ])
]

for model_name, layers in models_to_visualize:
    plot_model_diagram(model_name, layers)
    plt.show()

## 4. Feature Extraction Approach

In the feature extraction approach, we:

1. Take a pre-trained CNN model (typically trained on ImageNet).
2. Remove the original classifier (top layers).
3. Freeze all the remaining convolutional layers so their weights won't be updated.
4. Add our own new classifier on top that is specific to our task.
5. Train only the new classifier layers on our dataset.

This approach is useful when:
- Your new dataset is small
- Your new dataset is similar to the original dataset used to train the pre-trained model
- You want to avoid overfitting by not modifying the convolutional features

Let's implement feature extraction using TensorFlow/Keras:

In [ ]:
# Example: Feature extraction with VGG16 on a simple dataset

# Let's use CIFAR-10 as our small dataset example
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Normalize pixel values
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Convert class vectors to binary class matrices (one-hot encoding)
num_classes = 10
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

# CIFAR images are 32x32 but VGG16 expects at least 48x48 images
# We'll resize them to 48x48
x_train_resized = np.array([tf.image.resize(img, (48, 48)).numpy() for img in x_train])
x_test_resized = np.array([tf.image.resize(img, (48, 48)).numpy() for img in x_test])

print("Resized training data shape:", x_train_resized.shape)
print("Resized test data shape:", x_test_resized.shape)

In [ ]:
# Feature extraction using VGG16
# Load VGG16 with weights pre-trained on ImageNet
# We exclude the top (fully connected) layers
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(48, 48, 3))

# Freeze all layers in the base model
for layer in base_model.layers:
    layer.trainable = False

# Add our own classifier on top
model = keras.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

# Compile the model
model.compile(
    optimizer=keras.optimizers.legacy.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Print model summary
model.summary()

In [ ]:
# Train the model (we're using a small subset for demonstration purposes)
# In a real scenario, you would use the full dataset
subset_size = 5000  # Using a small subset for quick demonstration
history = model.fit(
    x_train_resized[:subset_size],
    y_train[:subset_size],
    batch_size=64,
    epochs=5,
    validation_data=(x_test_resized[:1000], y_test[:1000]),
    verbose=1
)

In [ ]:
# Plot training & validation accuracy and loss
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.tight_layout()
plt.show()

## 5. Fine-Tuning Approach

In the fine-tuning approach, we:

1. Take a pre-trained model and replace the top classifier layers with our own.
2. Initially train just the new classifier layers (like in feature extraction).
3. Then, "unfreeze" some of the later layers in the base model.
4. Continue training with a very low learning rate to fine-tune these layers to our specific task.

This approach is useful when:
- Your dataset is larger or more different from the original dataset
- You want to adapt the pre-trained features to better fit your specific task

Let's implement fine-tuning using TensorFlow/Keras:

In [ ]:
# Fine-tuning with ResNet50

# Load ResNet50 with pre-trained weights
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(48, 48, 3))

# First, freeze all layers
for layer in base_model.layers:
    layer.trainable = False

# Create our model with custom classifier
model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])

# Compile the model
model.compile(
    optimizer=keras.optimizers.legacy.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Phase 1: Train only the top layers (the classifier we added)
print("Phase 1: Training the top-level classifier...")
history_phase1 = model.fit(
    x_train_resized[:subset_size],
    y_train[:subset_size],
    batch_size=64,
    epochs=3,
    validation_data=(x_test_resized[:1000], y_test[:1000]),
    verbose=1
)

In [ ]:
# Phase 2: Unfreeze some layers for fine-tuning
# Let's unfreeze the last 4 layers of the ResNet50 base
for layer in base_model.layers[-4:]:
    layer.trainable = True

# Recompile with a lower learning rate
model.compile(
    optimizer=keras.optimizers.legacy.Adam(learning_rate=0.0001),  # Much lower learning rate
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Print which layers are now trainable
print("Trainable layers:")
for layer in model.layers[0].layers:
    print(f"{layer.name}: {layer.trainable}")

In [ ]:
# Phase 2: Fine-tune the model
print("\nPhase 2: Fine-tuning the model...")
history_phase2 = model.fit(
    x_train_resized[:subset_size],
    y_train[:subset_size],
    batch_size=32,  # Smaller batch size
    epochs=5,
    validation_data=(x_test_resized[:1000], y_test[:1000]),
    verbose=1
)

In [ ]:
# Plot the combined training history
plt.figure(figsize=(12, 4))

# Combine histories
combined_acc = history_phase1.history['accuracy'] + history_phase2.history['accuracy']
combined_val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']
combined_loss = history_phase1.history['loss'] + history_phase2.history['loss']
combined_val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']

# Plot accuracy
plt.subplot(1, 2, 1)
plt.plot(combined_acc)
plt.plot(combined_val_acc)
plt.axvline(x=len(history_phase1.history['accuracy'])-0.5, color='r', linestyle='--')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation', 'Fine-tuning starts'], loc='lower right')

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(combined_loss)
plt.plot(combined_val_loss)
plt.axvline(x=len(history_phase1.history['loss'])-0.5, color='r', linestyle='--')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation', 'Fine-tuning starts'], loc='upper right')

plt.tight_layout()
plt.show()

## 6. Implementing Transfer Learning with PyTorch

Now let's see how to implement transfer learning using PyTorch. We'll follow similar steps but using PyTorch's framework and APIs.

In [ ]:
# Check if GPU is available
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# PyTorch transformations
transform = transforms.Compose([
    transforms.Resize(224),  # ResNet requires 224x224 input
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # ImageNet normalization
])

# Load a small subset of CIFAR-10 for demonstration
# In a real project, you would use your own dataset
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Use smaller subsets for demonstration
subset_size = 5000
indices = list(range(subset_size))
trainset_subset = torch.utils.data.Subset(trainset, indices)
testset_subset = torch.utils.data.Subset(testset, indices[:1000])

# Create data loaders
trainloader = torch.utils.data.DataLoader(trainset_subset, batch_size=64, shuffle=True)
testloader = torch.utils.data.DataLoader(testset_subset, batch_size=64, shuffle=False)

# Classes in CIFAR-10
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [ ]:
# Feature Extraction with PyTorch using ResNet18

# Load pre-trained ResNet18
model_ft = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Freeze all parameters
for param in model_ft.parameters():
    param.requires_grad = False

# Replace the final fully connected layer
num_ftrs = model_ft.fc.in_features
model_ft.fc = nn.Linear(num_ftrs, 10)  # 10 classes in CIFAR-10

# Move model to the right device
model_ft = model_ft.to(device)

# Loss function and optimizer
criterion = nn.CrossEntropyLoss()

# Only optimize parameters of the final layer (which are not frozen)
optimizer_ft = optim.Adam(model_ft.fc.parameters(), lr=0.001)

# Print model structure
print(model_ft)

In [ ]:
# Training function
def train_model(model, criterion, optimizer, dataloader, epochs=5):
    model.train()
    train_loss = []
    train_acc = []
    
    for epoch in range(epochs):
        running_loss = 0.0
        running_corrects = 0
        processed_size = 0
        
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            # Zero the parameter gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            # Statistics
            batch_size = inputs.size(0)
            running_loss += loss.item() * batch_size
            running_corrects += torch.sum(preds == labels.data)
            processed_size += batch_size
        
        epoch_loss = running_loss / processed_size
        epoch_acc = running_corrects.double() / processed_size
        train_loss.append(epoch_loss)
        train_acc.append(epoch_acc.cpu().numpy())
        
        print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}")
    
    return train_loss, train_acc

# Evaluate function
def evaluate_model(model, criterion, dataloader):
    model.eval()
    running_loss = 0.0
    running_corrects = 0
    processed_size = 0
    
    with torch.no_grad():  # No gradients needed for evaluation
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            # Forward pass
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)
            
            # Statistics
            batch_size = inputs.size(0)
            running_loss += loss.item() * batch_size
            running_corrects += torch.sum(preds == labels.data)
            processed_size += batch_size
    
    loss = running_loss / processed_size
    acc = running_corrects.double() / processed_size
    
    print(f"Evaluation - Loss: {loss:.4f}, Acc: {acc:.4f}")
    return loss, acc.item()

In [ ]:
# Train the model (feature extraction)
print("Phase 1: Training only the classifier layer...")
feature_extraction_loss, feature_extraction_acc = train_model(
    model_ft, criterion, optimizer_ft, trainloader, epochs=3
)

# Evaluate the model
print("\nEvaluating after feature extraction...")
val_loss, val_acc = evaluate_model(model_ft, criterion, testloader)

In [ ]:
# Fine-tuning: Unfreeze some layers
print("\nPhase 2: Fine-tuning - unfreezing some layers...")

# Unfreeze the last two layers of the ResNet
for name, child in list(model_ft.named_children())[-2:]:
    for param in child.parameters():
        param.requires_grad = True

# New optimizer with lower learning rate for fine-tuning
optimizer_ft = optim.Adam([{
    'params': [p for p in model_ft.parameters() if p.requires_grad]
}], lr=0.0001)

# Train with fine-tuning
fine_tuning_loss, fine_tuning_acc = train_model(
    model_ft, criterion, optimizer_ft, trainloader, epochs=3
)

# Evaluate after fine-tuning
print("\nEvaluating after fine-tuning...")
val_loss_ft, val_acc_ft = evaluate_model(model_ft, criterion, testloader)

In [ ]:
# Plot the results
plt.figure(figsize=(12, 4))

# Plot accuracy
plt.subplot(1, 2, 1)
plt.plot(feature_extraction_acc + fine_tuning_acc)
plt.axvline(x=len(feature_extraction_acc)-0.5, color='r', linestyle='--')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train Accuracy', 'Fine-tuning starts'])

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(feature_extraction_loss + fine_tuning_loss)
plt.axvline(x=len(feature_extraction_loss)-0.5, color='r', linestyle='--')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train Loss', 'Fine-tuning starts'])

plt.tight_layout()
plt.show()

## 7. Transfer Learning for Different Domains

Transfer learning can be applied to different domains beyond image classification. Let's discuss various applications:

### Text and Natural Language Processing

In NLP, transfer learning has revolutionized the field with models like:

1. **BERT (Bidirectional Encoder Representations from Transformers)**
   - Pre-trained on massive text corpora
   - Can be fine-tuned for specific NLP tasks like sentiment analysis, text classification, etc.

2. **GPT (Generative Pre-trained Transformer) family**
   - Pre-trained for language modeling, then fine-tuned for specific tasks
   - Used for text generation, translation, summarization, etc.

3. **Word Embeddings (Word2Vec, GloVe, FastText)**
   - Transfer learning at the feature level
   - Pre-trained word vectors that capture semantic meaning

### Audio and Speech Processing

1. **Wav2Vec, HuBERT, etc.**
   - Pre-trained on large audio datasets
   - Fine-tuned for speech recognition, speaker identification, etc.

2. **Music and Audio Classification**
   - Models pre-trained on general audio can be fine-tuned for specific sound classification

### Time Series Analysis

1. **Pre-trained models for general time-series patterns**
   - Fine-tuned for specific forecasting tasks
   - Useful in finance, IoT, weather forecasting, etc.

### Medical Imaging

1. **Models pre-trained on natural images**
   - Fine-tuned for medical image analysis (X-rays, MRIs, etc.)
   - Disease detection, organ segmentation, etc.

## 8. Visualization of Transfer Learning Results

Let's create some visualizations to better understand what happens during transfer learning:

In [ ]:
# Let's visualize activations in different layers of our transfer learning model
# First, let's get a few test images

# For TensorFlow/Keras
def visualize_layer_activations(model, image, layer_name):
    """Visualize the activations of a specific layer for a given image"""
    # Create a model that outputs the activations of the specified layer
    layer_model = keras.Model(inputs=model.input, outputs=model.get_layer(layer_name).output)
    activations = layer_model.predict(np.expand_dims(image, axis=0))
    
    # Plot a subset of activation channels
    plt.figure(figsize=(12, 6))
    plt.suptitle(f"Activations in layer: {layer_name}")
    
    # Get the number of channels (filters) in the activation output
    channels = activations.shape[-1]
    n_cols = 8
    n_rows = min(4, (channels + n_cols - 1) // n_cols)
    total_plots = min(n_rows * n_cols, channels)
    
    # Plot each channel
    for i in range(total_plots):
        plt.subplot(n_rows, n_cols, i+1)
        plt.imshow(activations[0, :, :, i], cmap='viridis')
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# Get a sample image
sample_idx = 5  # Just pick one example
sample_image = x_test_resized[sample_idx]
sample_label = np.argmax(y_test[sample_idx])

# Display the sample image
plt.figure(figsize=(4, 4))
plt.imshow(sample_image)
plt.title(f"Sample Image (class: {sample_label})")
plt.axis('off')
plt.show()

In [ ]:
# Visualize activations from early, middle and late convolutional layers
# Note: Use actual layer names from your model (may differ from these examples)
try:
    early_layer = 'block1_conv1'  # Example for VGG16
    middle_layer = 'block3_conv1'
    late_layer = 'block5_conv1'
    
    print("Early layer activations:")
    visualize_layer_activations(model, sample_image, early_layer)
    
    print("Middle layer activations:")
    visualize_layer_activations(model, sample_image, middle_layer)
    
    print("Late layer activations:")
    visualize_layer_activations(model, sample_image, late_layer)
except Exception as e:
    print(f"Error visualizing activations: {e}")
    print("This might be because the layer names don't match your model.")
    print("Available layers in the model:")
    for i, layer in enumerate(model.layers[0].layers[:10]):
        print(f"{i}: {layer.name}")
    print("...and more")

## 9. Common Challenges and Best Practices in Transfer Learning

### Challenges

1. **Domain Mismatch**
   - Pre-trained model was trained on a domain very different from target domain
   - Example: ImageNet (natural images) vs. medical images

2. **Catastrophic Forgetting**
   - During fine-tuning, model may forget useful features from pre-training

3. **Layer Freezing Strategy**
   - Deciding which layers to freeze and which to fine-tune

4. **Hyperparameter Tuning**
   - Finding optimal learning rates, especially during fine-tuning

### Best Practices

1. **Choose the Right Pre-trained Model**
   - Select models pre-trained on data similar to your target domain
   - Consider model size vs. computational resources tradeoff

2. **Progressive Fine-Tuning**
   - Start by training only the classifier
   - Gradually unfreeze and fine-tune deeper layers
   - Use lower learning rates for pre-trained layers

3. **Learning Rate Strategies**
   - Use lower learning rates for pre-trained layers
   - Higher learning rates for newly added layers
   - Consider learning rate schedulers

4. **Data Augmentation**
   - Especially important with small datasets
   - Helps prevent overfitting during fine-tuning

5. **Monitor Performance**
   - Watch for signs of overfitting
   - Use early stopping where appropriate
   
6. **Layer Freezing Strategy**
   - Earlier layers learn more generic features (edges, textures)
   - Later layers learn more specific features (objects, patterns)
   - Generally, freeze early layers and fine-tune later layers

In [ ]:
# Create a visualization of best practices for transfer learning
plt.figure(figsize=(12, 8))
plt.axis('off')

# Title
plt.text(0.5, 0.95, "Transfer Learning Best Practices", fontsize=20, ha='center', fontweight='bold')

# Left column - When to use feature extraction
plt.text(0.25, 0.85, "Feature Extraction", fontsize=16, ha='center', fontweight='bold')
plt.text(0.25, 0.80, "When to use:", fontsize=14, ha='center', fontweight='bold')
plt.text(0.25, 0.76, "• Small dataset", fontsize=12, ha='left')
plt.text(0.25, 0.73, "• Target domain similar to source", fontsize=12, ha='left')
plt.text(0.25, 0.70, "• Limited computational resources", fontsize=12, ha='left')
plt.text(0.25, 0.67, "• Fast development needed", fontsize=12, ha='left')

# Right column - When to use fine-tuning
plt.text(0.75, 0.85, "Fine-Tuning", fontsize=16, ha='center', fontweight='bold')
plt.text(0.75, 0.80, "When to use:", fontsize=14, ha='center', fontweight='bold')
plt.text(0.75, 0.76, "• Larger dataset available", fontsize=12, ha='left')
plt.text(0.75, 0.73, "• Target domain differs from source", fontsize=12, ha='left')
plt.text(0.75, 0.70, "• Better performance needed", fontsize=12, ha='left')
plt.text(0.75, 0.67, "• More resources available", fontsize=12, ha='left')

# Lower section - General best practices
plt.text(0.5, 0.55, "General Best Practices", fontsize=16, ha='center', fontweight='bold')

practices = [
    "1. Start with a proper pre-trained model (consider domain similarity)",
    "2. Data preprocessing should match the pre-trained model's training data",
    "3. Use progressive fine-tuning (train classifier first, then unfreeze gradually)",
    "4. Use smaller learning rates when fine-tuning pre-trained layers",
    "5. Apply data augmentation to prevent overfitting",
    "6. Monitor validation performance to avoid overfitting",
    "7. Early layers capture generic features; later layers are more specific",
    "8. Consider layer freezing strategy based on dataset size and similarity"
]

y_pos = 0.5
for practice in practices:
    plt.text(0.1, y_pos, practice, fontsize=12)
    y_pos -= 0.04

# Common pitfalls
plt.text(0.5, 0.15, "Common Pitfalls to Avoid", fontsize=16, ha='center', fontweight='bold')
pitfalls = [
    "• Using incompatible input dimensions or preprocessing",
    "• Fine-tuning with too high learning rate (catastrophic forgetting)",
    "• Not freezing enough layers with very small datasets",
    "• Neglecting data augmentation with small datasets",
    "• Using inappropriate pre-trained models for your domain"
]

y_pos = 0.1
for pitfall in pitfalls:
    plt.text(0.1, y_pos, pitfall, fontsize=12)
    y_pos -= 0.03

plt.tight_layout()
plt.show()

## Summary

In this notebook, we've explored transfer learning, which allows us to leverage pre-trained models to achieve better performance on new tasks with less data and computational resources.

Key takeaways:

1. **What is Transfer Learning?**
   - Reusing knowledge from pre-trained models for new tasks

2. **Two Main Approaches:**
   - Feature extraction: Freeze pre-trained model, add new classifier
   - Fine-tuning: Adapt some or all layers of pre-trained model

3. **Popular Pre-trained Models:**
   - Vision: VGG, ResNet, MobileNet, EfficientNet
   - Text: BERT, GPT, Word Embeddings
   - Various architectures for different domains

4. **Implementation in Major Frameworks:**
   - TensorFlow/Keras and PyTorch provide simple APIs for transfer learning

5. **Best Practices:**
   - Choose appropriate pre-trained models
   - Use progressive fine-tuning
   - Apply proper learning rates
   - Use data augmentation

6. **Applications Beyond Vision:**
   - NLP, audio processing, time series, medical imaging, etc.

Transfer learning has revolutionized deep learning, making it practical to deploy sophisticated models with limited data and computational resources. It's an essential technique in any machine learning practitioner's toolkit.